# 🎬 أداة فيديو أفق

أداة مستقلة 100% لتوليد فيديوهات إعلانية احترافية.

**خطوات الاستخدام:**
1. شغّل كل الخلايا بالترتيب (Run All)
2. عدّل الوصف في الخلية 5
3. شغّل الخلية 6
4. حمّل الفيديو من مجلد output/

**المتطلبات:** حساب Google (مجاني)

> **تنبيه:** تم ربط الخلية 5 بالـCLI دون إدخال تفاعلي، لكن Run All لم يُختبر على Colab بعد. يلزم أيضًا إعداد SadTalker وصورة مصرّح بها ومتصفح وخط عربي وفق README؛ الخلية 4 وحدها لا تكمل هذه المتطلبات.


In [ ]:
!nvidia-smi
!ffmpeg -version | head -1
!python --version
!node --version


In [ ]:
import os
import subprocess

REPO = "https://github.com/amjadalobd1234-blip/ofoq-agency.git"
PROJECT_DIR = "/content/ofoq-agency"

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO} {PROJECT_DIR}
else:
    !cd {PROJECT_DIR} && git pull

os.chdir(f"{PROJECT_DIR}/tools/video-agent")
print(f"✅ Working dir: {os.getcwd()}")
!ls -la


In [ ]:
# النظام
!apt-get install -y -qq gfortran ffmpeg

# Node
!npm install

# Python
!pip install -q piper-tts numpy scipy librosa soundfile
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121

# تحميل نموذج Piper العربي
!mkdir -p models/piper
![ -f models/piper/ar_JO-kareem-medium.onnx ] || \
    wget -q https://huggingface.co/rhasspy/piper-voices/resolve/main/ar/ar_JO/kareem/medium/ar_JO-kareem-medium.onnx -O models/piper/ar_JO-kareem-medium.onnx
![ -f models/piper/ar_JO-kareem-medium.onnx.json ] || \
    wget -q https://huggingface.co/rhasspy/piper-voices/resolve/main/ar/ar_JO/kareem/medium/ar_JO-kareem-medium.onnx.json -O models/piper/ar_JO-kareem-medium.onnx.json

print("✅ تم التثبيت")


In [ ]:
# ✏️ عدّل هذا الوصف
DESCRIPTION = "فيديو إعلاني لعطر ذهبي فاخر، 15 ثانية"

# اختر القالب
TEMPLATE = "perfume"  # perfume / restaurant / clothing / services

# معلومات اختيارية
PRODUCT_NAME = "عطر الذهب"
BENEFIT = "يدوم 24 ساعة"
PRICE = "50,000 دينار"

print(f"📝 الوصف: {DESCRIPTION}")
print(f"📁 القالب: {TEMPLATE}")


In [ ]:
import os
import subprocess
from pathlib import Path
from IPython.display import Video, display

working_dir = Path(PROJECT_DIR) / "tools/video-agent"
output_dir = working_dir / "output"
previous_files = set(output_dir.glob("*/ad-*.mp4"))

# تمرير كل قيمة كـ argument مستقل يحافظ على العربية والاقتباسات دون shell.
# الفاصل -- ضروري لتمرير الخيارات من npm إلى الأداة.
command = [
    "npm", "start", "--",
    "--description", DESCRIPTION,
    "--template", TEMPLATE,
    "--productName", PRODUCT_NAME,
    "--benefit", BENEFIT,
    "--price", PRICE,
    "--non-interactive",
]
env = os.environ.copy()
env["OUTPUT_DIR"] = str(output_dir)
result = subprocess.run(
    command,
    cwd=working_dir,
    env=env,
    stdin=subprocess.DEVNULL,
    capture_output=True,
    text=True,
    timeout=3600,
)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)
# لا نعرض فيديو قديم أو رسالة نجاح إذا فشل هذا التشغيل.
result.check_returncode()

files = sorted(
    set(output_dir.glob("*/ad-*.mp4")) - previous_files,
    key=lambda file: file.stat().st_mtime,
)
if not files:
    raise FileNotFoundError("انتهى الأمر دون إنشاء فيديو نهائي جديد في output/")
video_path = files[-1]
print(f"✅ الفيديو: {video_path}")
print("حمّل الفيديو من لوحة Files في Colab داخل مجلد output/ الخاص بالأداة.")
display(Video(str(video_path), embed=True))
